# Práctica Final TI24 - Preprocesamiento de Datos
## Tema: Bagging - Random Forests
### Dataset: Loan Default Prediction


Se aplican las siguientes etapas:
- Limpieza de datos (eliminación de columnas irrelevantes, verificación de nulos y duplicados)
- Codificación (encoding) de variables categóricas
- Escalado (scaling) de variables numéricas
- División del dataset en conjuntos de entrenamiento y prueba (train/test split)

El resultado de este notebook es un dataset listo para entrenar los modelos
en `03_model_main.ipynb` y `04_model_comparison.ipynb`.


## 1. Importación de librerías

Se importan las librerías necesarias para manipulación de datos, preprocesamiento
y división del dataset.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import joblib


## 2. Carga del dataset

Se carga el dataset desde el repositorio de GitHub del proyecto, de la misma
forma que en el notebook de EDA, para garantizar reproducibilidad.


In [2]:
url = "https://raw.githubusercontent.com/soledadaldunate/repositorio-TI24-Aldunate/main/data/raw/Loan_default.csv"
df = pd.read_csv(url)
print("Dimensiones originales:", df.shape)
df.head()


Dimensiones originales: (255347, 18)


,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


## 3. Eliminación de columnas irrelevantes

La columna `LoanID` es un identificador único de cada registro y no aporta
información predictiva para el modelo, por lo que se elimina.


In [3]:
df = df.drop(columns=["LoanID"])
print("Dimensiones tras eliminar LoanID:", df.shape)
df.head()


Dimensiones tras eliminar LoanID: (255347, 17)


,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default
0,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0
1,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0
2,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1
3,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0
4,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0


## 4. Verificación de valores nulos y duplicados

Tal como se identificó en el EDA, el dataset no presenta valores nulos. Sin
embargo, se vuelve a verificar aquí como buena práctica antes de continuar
con el preprocesamiento, y se revisa también la existencia de registros
duplicados.


In [4]:
print("Valores nulos por columna:")
print(df.isnull().sum().sum(), "valores nulos en total")

print("\nRegistros duplicados:", df.duplicated().sum())


Valores nulos por columna:
0 valores nulos en total

Registros duplicados: 0


**Interpretación:** Al no existir valores nulos ni duplicados, no es
necesario aplicar técnicas de imputación ni de eliminación de registros. El
dataset puede continuar directamente hacia la etapa de codificación.


## 5. Codificación de variables categóricas (Encoding)

Se identifican las variables categóricas del dataset. Algunas son binarias
(Yes/No) y otras tienen múltiples categorías (nominales). Para este proyecto
se utiliza **Label Encoding**, dado que Random Forest (y los modelos basados
en árboles en general) no requieren que las variables categóricas estén en
formato one-hot, ya que los árboles dividen el espacio en función de umbrales
sobre cada variable, sin asumir relaciones de orden o distancia entre
categorías.


In [5]:
variables_categoricas = ["Education", "EmploymentType", "MaritalStatus",
                         "HasMortgage", "HasDependents", "LoanPurpose",
                         "HasCoSigner"]

encoders = {}

for col in variables_categoricas:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")


Education: {"Bachelor's": np.int64(0), 'High School': np.int64(1), "Master's": np.int64(2), 'PhD': np.int64(3)}
EmploymentType: {'Full-time': np.int64(0), 'Part-time': np.int64(1), 'Self-employed': np.int64(2), 'Unemployed': np.int64(3)}
MaritalStatus: {'Divorced': np.int64(0), 'Married': np.int64(1), 'Single': np.int64(2)}
HasMortgage: {'No': np.int64(0), 'Yes': np.int64(1)}
HasDependents: {'No': np.int64(0), 'Yes': np.int64(1)}
LoanPurpose: {'Auto': np.int64(0), 'Business': np.int64(1), 'Education': np.int64(2), 'Home': np.int64(3), 'Other': np.int64(4)}
HasCoSigner: {'No': np.int64(0), 'Yes': np.int64(1)}


**Interpretación:** Cada categoría textual fue reemplazada por un valor
numérico entero. Por ejemplo, las variables binarias `HasMortgage`,
`HasDependents` y `HasCoSigner` quedan codificadas como 0 (No) y 1 (Yes), y
las variables con múltiples categorías (`Education`, `EmploymentType`,
`MaritalStatus`, `LoanPurpose`) quedan codificadas con valores de 0 en
adelante según el orden alfabético de sus categorías.


## 6. Separación de variables predictoras (X) y variable objetivo (y)

Se separa el dataset en la matriz de variables predictoras `X` y el vector
de la variable objetivo `y` (`Default`).


In [6]:
X = df.drop(columns=["Default"])
y = df["Default"]

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)
print("\nVariables predictoras:")
print(list(X.columns))


Forma de X: (255347, 16)
Forma de y: (255347,)

Variables predictoras:
['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education', 'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']


## 7. División en conjuntos de entrenamiento y prueba (train/test split)

Se divide el dataset en 80% para entrenamiento y 20% para prueba. Se utiliza
el parámetro `stratify=y` para mantener la misma proporción de clases
(88.4% / 11.6%) observada en el EDA en ambos conjuntos, dado que se trata de
un problema con clases desbalanceadas.


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_test:", X_test.shape)

print("\nDistribución de clases en y_train:")
print(y_train.value_counts(normalize=True).round(3) * 100)

print("\nDistribución de clases en y_test:")
print(y_test.value_counts(normalize=True).round(3) * 100)


Tamaño de X_train: (204277, 16)
Tamaño de X_test: (51070, 16)

Distribución de clases en y_train:
Default
0    88.4
1    11.6
Name: proportion, dtype: float64

Distribución de clases en y_test:
Default
0    88.4
1    11.6
Name: proportion, dtype: float64


**Interpretación:** La estratificación garantiza que tanto el conjunto
de entrenamiento como el de prueba mantengan la proporción original de
clases (aprox. 88% sin incumplimiento y 12% con incumplimiento), evitando que
el split introduzca un sesgo adicional sobre el desbalance ya existente.


In [8]:
variables_numericas = ["Age", "Income", "LoanAmount", "CreditScore",
                       "MonthsEmployed", "NumCreditLines", "InterestRate",
                       "LoanTerm", "DTIRatio"]

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[variables_numericas] = scaler.fit_transform(X_train[variables_numericas])
X_test_scaled[variables_numericas] = scaler.transform(X_test[variables_numericas])

X_train_scaled.head()


,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner
15826,0.099486,-1.167307,1.698627,0.312274,-1.170591,-0.449335,-1.335928,1.414459,1.512942,1,3,1,0,1,0,0
147371,0.299620,1.319747,-0.864170,-0.505800,1.714171,0.445947,0.185568,0.707489,-0.045473,3,2,1,1,1,4,1
178180,0.232908,0.453497,-1.700954,0.903803,1.396847,0.445947,-1.201856,-0.706451,1.123338,1,2,2,1,1,1,1
126915,-0.100649,-1.191966,-1.432895,-1.449730,-1.661000,0.445947,0.723364,0.000519,1.123338,0,2,1,0,1,2,0
163930,-1.568299,0.434508,1.707671,-1.613345,0.416028,0.445947,0.898110,1.414459,-0.218630,0,1,0,0,0,0,1


## 9. Resumen del preprocesamiento

Se resume el resultado final del preprocesamiento, dejando los conjuntos
listos para el entrenamiento de los modelos.


In [9]:
print("Resumen del preprocesamiento")
print("="*40)
print("Filas totales:", df.shape[0])
print("Columnas predictoras:", X.shape[1])
print("Variables categóricas codificadas (Label Encoding):", len(variables_categoricas))
print("Variables numéricas escaladas (StandardScaler):", len(variables_numericas))
print("\nX_train_scaled:", X_train_scaled.shape)
print("X_test_scaled :", X_test_scaled.shape)
print("y_train       :", y_train.shape)
print("y_test        :", y_test.shape)


Resumen del preprocesamiento
Filas totales: 255347
Columnas predictoras: 16
Variables categóricas codificadas (Label Encoding): 7
Variables numéricas escaladas (StandardScaler): 9

X_train_scaled: (204277, 16)
X_test_scaled : (51070, 16)
y_train       : (204277,)
y_test        : (51070,)


## 10. Guardado de los datos procesados

Se guardan los conjuntos de entrenamiento y prueba en la carpeta
`data/processed/`, junto con el escalador y los codificadores, para ser
reutilizados directamente en los notebooks de modelado (`03_model_main.ipynb`
y `04_model_comparison.ipynb`) sin tener que repetir el preprocesamiento.


In [10]:
import os

os.makedirs("data/processed", exist_ok=True)

X_train_scaled.to_csv("data/processed/X_train.csv", index=False)
X_test_scaled.to_csv("data/processed/X_test.csv", index=False)
y_train.to_csv("data/processed/y_train.csv", index=False)
y_test.to_csv("data/processed/y_test.csv", index=False)

joblib.dump(scaler, "data/processed/scaler.pkl")
joblib.dump(encoders, "data/processed/encoders.pkl")

print("Archivos guardados en data/processed/")


Archivos guardados en data/processed/
